In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import numpy as np

c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Knowledge base
docs = [
    "AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.",
    "The system can use satellite data, weather data, water temperature, and sensor measurements.",
    "Dense retrieval uses embeddings to find documents with similar meaning.",
    "BM25 is a sparse retrieval method based on keyword matching.",
    "Transformers are neural network architectures used in models such as GPT, BERT, and LLaMA."
]

query = "How does AI4HAB predict algae blooms?"

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedding_model.encode(docs)
query_embedding = embedding_model.encode([query])

# Similarity search
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k = 2
top_indices = np.argsort(similarities)[::-1][:top_k]
retrieved_docs = [docs[i] for i in top_indices]

print("Retrieved Documents:\n")
for doc in retrieved_docs:
    print("-", doc)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2576.06it/s]


Retrieved Documents:

- AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.
- The system can use satellite data, weather data, water temperature, and sensor measurements.


In [4]:

# Build context
context = " ".join(retrieved_docs)

# This is what LLM will see as input
# Improved prompt
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Write one clear sentence.
Answer:
"""

# Better text generation model
generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

answer = generator(
    prompt,
    max_new_tokens=80,
    do_sample=False 
)[0]["generated_text"]

print("\nGenerated Answer:\n")
print(answer)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2679.06it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo


Generated Answer:


Answer the question using only the context below.

Context:
AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes. The system can use satellite data, weather data, water temperature, and sensor measurements.

Question:
How does AI4HAB predict algae blooms?

Write one clear sentence.
Answer:

